# NMOS-input folded-cascode OTA

Drives the gate of the PMOS regulated-cascode device T5, so its output sits at V_ref2 - VSG(T5). Every device VDS follows from the node plan in `specs.ota_n`.

In [1]:
import math
import pandas as pd
from sarlib import specs as S
from sarlib.gmid import nmos, pmos, size, id_w

o = S.ota_n
I_PAIR = o.I_TAIL / 2
I_TOP = o.TOP_RATIO * o.I_TAIL
I_BRANCH = I_TOP - I_PAIR
sz = lambda mos, key, ID, VDS, VSB=0.0: size(mos, ID, o.GMID[key], o.LEN[key], VDS, VSB)

## Output range set by T5

In [2]:
i_mid = 0.5 * sum(o.I_T5)
t5 = sz(pmos, 't5', i_mid, 0.3, S.VDD - o.VCM)
V_OUT_HI = o.VCM - (t5['VGS'] + (o.I_T5[0] - i_mid) / t5['gm'])
V_OUT_LO = o.VCM - (t5['VGS'] + (o.I_T5[1] - i_mid) / t5['gm'])
print(f"T5: W = {t5['W']:.1f} um, VSG = {t5['VGS']:.3f} V at {i_mid:.0f} uA")
print(f'OTA output must cover {V_OUT_LO:.3f} to {V_OUT_HI:.3f} V')

T5: W = 36.2 um, VSG = 1.190 V at 30 uA
OTA output must cover 0.332 to 0.387 V


## Devices

In [3]:
vx = 0.3
for _ in range(20):
    pair = sz(nmos, 'pair', I_PAIR, o.V_FOLD - vx, vx)
    vx, done = o.VCM - pair['VGS'], abs(o.VCM - pair['VGS'] - vx) < 1e-3
    if done:
        break
pair = sz(nmos, 'pair', I_PAIR, o.V_FOLD - vx, vx)
tail = sz(nmos, 'tail', o.I_TAIL, vx)
mir = sz(nmos, 'mirror', I_BRANCH, o.V_MIR)
nc = sz(nmos, 'ncasc', I_BRANCH, V_OUT_LO - o.V_MIR, o.V_MIR)
pc = sz(pmos, 'pcasc', I_BRANCH, o.V_FOLD - V_OUT_HI, S.VDD - o.V_FOLD)
pt = sz(pmos, 'ptop', I_TOP, S.VDD - o.V_FOLD)
V_DIODE = mir['VGS']

devs = [('M2/M3 pair', pair, False), ('M1 tail', tail, False), ('M4/M5 mirror', mir, True),
        ('M10/M11 n cascode', nc, True), ('M6/M7 p cascode', pc, True), ('M8/M9 p source', pt, False)]
rows = []
for name, d, branch in devs:
    m = d['VDS'] - d['VDSAT']
    m_top = d['VDS'] - d['VDSAT'] * math.sqrt(I_TOP / d['ID']) if branch else m   # pair fully switched
    rows.append(dict(device=name, ID=d['ID'], W=d['W'], VGS=d['VGS'], VSB=d['VSB'], VDS=d['VDS'], VDSAT=d['VDSAT'],
                     margin=m, margin_at_I_TOP=m_top))
pd.DataFrame(rows).set_index('device').round(3)

,ID,W,VGS,VSB,VDS,VDSAT,margin,margin_at_I_TOP
device,,,,,,,,
M2/M3 pair,10.0,7.563,0.832,0.719,0.781,0.139,0.642,0.642
M1 tail,20.0,5.321,0.798,0.000,0.719,0.219,0.500,0.500
M4/M5 mirror,14.0,13.975,0.681,0.000,0.180,0.128,0.052,0.013
M10/M11 n cascode,14.0,26.627,0.664,0.180,0.152,0.096,0.056,0.026
M6/M7 p cascode,14.0,12.826,1.329,0.300,1.113,0.225,0.887,0.817
M8/M9 p source,24.0,26.299,1.248,0.000,0.300,0.210,0.090,0.090


In [4]:
bias = dict(Vbiasn1=tail['VGS'], Vbiasn2=o.V_MIR + nc['VGS'], Vbiasp1=S.VDD - pt['VGS'], Vbiasp2=o.V_FOLD - pc['VGS'])
out_lo, out_hi = o.V_MIR + nc['VDSAT'] + o.MARGIN, o.V_FOLD - pc['VDSAT'] - o.MARGIN
d_lo, d_hi = S.VREF1 + o.VDSAT_T1 + o.MARGIN, o.VCM - t5['VDSAT'] - o.MARGIN
print('bias: ' + ', '.join(f'{k} = {v:.3f} V' for k, v in bias.items()))
print(f'nodes: Vx = {vx:.3f}, fold = {o.V_FOLD:.2f}, mirror drain = {o.V_MIR:.3f}, mirror gate = {V_DIODE:.3f}')
print(f'output window {out_lo:.2f} to {out_hi:.2f} V, needed {V_OUT_LO:.2f} to {V_OUT_HI:.2f} V: '
      + ('inside' if out_lo <= V_OUT_LO and V_OUT_HI <= out_hi else 'OUTSIDE'))
print(f'delta-I node window {d_lo:.2f} to {d_hi:.2f} V (centre {0.5 * (d_lo + d_hi):.2f} V)')
print(f'currents: tail {o.I_TAIL:g}, top {I_TOP:g}, branch {I_BRANCH:g} uA; supply {2 * I_TOP:.0f} uA')

bias: Vbiasn1 = 0.798 V, Vbiasn2 = 0.844 V, Vbiasp1 = 0.552 V, Vbiasp2 = 0.171 V
nodes: Vx = 0.719, fold = 1.50, mirror drain = 0.180, mirror gate = 0.681
output window 0.33 to 1.22 V, needed 0.33 to 0.39 V: inside
delta-I node window 0.50 to 1.35 V (centre 0.92 V)
currents: tail 20, top 24, branch 14 uA; supply 48 uA


## Cascode bias generators

Diode-connected devices whose VGS equals the cascode gate bias, at the branch current.

In [5]:
wn = I_BRANCH / id_w(nmos, bias['Vbiasn2'], bias['Vbiasn2'], o.LEN['ncasc'])
vsg_p = S.VDD - bias['Vbiasp2']
wp = I_BRANCH / id_w(pmos, vsg_p, vsg_p, o.LEN['pcasc'])
print(f"Vbiasn2 diode: nfet W = {wn:.2f} um, L = {o.LEN['ncasc']}")
print(f"Vbiasp2 diode: pfet W = {wp:.2f} um, L = {o.LEN['pcasc']}")

Vbiasn2 diode: nfet W = 2.66 um, L = 2
Vbiasp2 diode: pfet W = 2.29 um, L = 2


## DC gain and bandwidth

In [6]:
G = lambda r: r['gm'] * 1e-6
gm1 = G(pair)
r_fold = 1 / ((pair['gds'] + pt['gds']) * 1e-6)
ro_p, ro_nc, ro_mir = (1 / (d['gds'] * 1e-6) for d in (pc, nc, mir))
R_up = G(pc) * ro_p * r_fold + ro_p + r_fold
R_down = G(nc) * ro_nc * ro_mir + ro_nc + ro_mir
R_out = R_up * R_down / (R_up + R_down)
A0 = gm1 * R_out
print(f'R_up {R_up / 1e6:.2f} Mohm, R_down {R_down / 1e6:.2f} Mohm, R_out {R_out / 1e6:.2f} Mohm')
print(f'A0 = {A0:.0f} ({20 * math.log10(A0):.1f} dB), GBW = {gm1 / (2 * math.pi * o.C_LOAD) / 1e6:.1f} MHz, '
      f'dominant pole {1 / (2 * math.pi * R_out * o.C_LOAD) / 1e3:.1f} kHz')

R_up 108.92 Mohm, R_down 4.90 Mohm, R_out 4.69 Mohm
A0 = 656 (56.3 dB), GBW = 74.3 MHz, dominant pole 113.2 kHz
